In [ ]:
!pip install yfinance textstat vaderSentiment pdfplumber requests beautifulsoup4

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 1.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.0/71.0 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 177.1/177.1 kB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.0/126.0 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 91.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 102.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 70.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 79.6 MB/s eta 0:00:00
  Attempting uninstall: Pillow
    Found existing installation: pillow 11.3.0
    Uninstalling pillow-11.3.0:
      Successfully uninstalled pillow-11.3.0


In [ ]:
import requests
import pdfplumber
import pandas as pd
import numpy as np
import re
import io
import time
import yfinance as yf
from datetime import datetime, timedelta
from textstat import flesch_kincaid_grade
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

In [ ]:
fomc_dates = [
    # 2011 - Bernanke
    "20110427", "20110622", "20110921", "20111102",
    # 2012
    "20120125", "20120425", "20120620", "20120912", "20121212",
    # 2013
    "20130320", "20130619", "20130918", "20131218",
    # 2014 - Yellen
    "20140319", "20140618", "20140917", "20141217",
    # 2015
    "20150318", "20150617", "20150917", "20151216",
    # 2016
    "20160316", "20160615", "20160921", "20161214",
    # 2017
    "20170315", "20170614", "20170920", "20171213",
    # 2018 - Powell
    "20180321", "20180613", "20180926", "20181219",
    # 2019
    "20190130", "20190320", "20190501", "20190619", "20190731",
    "20190918", "20191030", "20191211",
    # 2020
    "20200129", "20200303", "20200315", "20200429", "20200610",
    "20200729", "20200916", "20201105", "20201216",
    # 2021
    "20210127", "20210317", "20210428", "20210616",
    "20210728", "20210922", "20211103", "20211215",
    # 2022
    "20220126", "20220316", "20220504", "20220615",
    "20220727", "20220921", "20221102", "20221214",
    # 2023
    "20230201", "20230322", "20230503", "20230614",
    "20230726", "20230920", "20231101", "20231213",
    # 2024
    "20240131", "20240320", "20240501", "20240612",
    "20240731", "20240918", "20241107", "20241218",
    # 2025
    "20250129", "20250319", "20250507", "20250618",
    "20250730", "20250917", "20251029", "20251210",
    # 2026
    "20260128", "20260318",
]

print(f"Total press conferences: {len(fomc_dates)}")

Total press conferences: 92


In [ ]:
def download_transcript(date_str):
    urls_to_try = [
        f"https://www.federalreserve.gov/mediacenter/files/FOMCpresconf{date_str}.pdf",
        f"https://www.federalreserve.gov/mediacenter/files/fomcpresconf{date_str}.pdf",
    ]

    for url in urls_to_try:
        try:
            response = requests.get(url, timeout=30)
            if response.status_code == 200:
                pdf_file = io.BytesIO(response.content)
                text = ""
                with pdfplumber.open(pdf_file) as pdf:
                    for page in pdf.pages:
                        page_text = page.extract_text()
                        if page_text:
                            text += page_text + "\n"
                print(f"  ✓ {date_str} - {len(text)} chars")
                return text
        except:
            continue

    print(f"  ✗ {date_str} - FAILED")
    return None

transcripts = {}
for date_str in fomc_dates:
    transcripts[date_str] = download_transcript(date_str)
    #time.sleep(1)

successful = sum(1 for v in transcripts.values() if v is not None)
print(f"\nGot {successful}/{len(fomc_dates)} transcripts")

  ✓ 20110427 - 52401 chars
  ✓ 20110622 - 43627 chars
  ✗ 20110921 - FAILED
  ✓ 20111102 - 44602 chars
  ✓ 20120125 - 60374 chars
  ✓ 20120425 - 45785 chars
  ✓ 20120620 - 42983 chars
  ✗ 20120912 - FAILED
  ✓ 20121212 - 68860 chars
  ✓ 20130320 - 52562 chars
  ✓ 20130619 - 55717 chars
  ✓ 20130918 - 54053 chars
  ✓ 20131218 - 62568 chars
  ✓ 20140319 - 46021 chars
  ✓ 20140618 - 42104 chars
  ✓ 20140917 - 44082 chars
  ✓ 20141217 - 45407 chars
  ✓ 20150318 - 42377 chars
  ✓ 20150617 - 44882 chars
  ✓ 20150917 - 45449 chars
  ✓ 20151216 - 52245 chars
  ✓ 20160316 - 45734 chars
  ✓ 20160615 - 41670 chars
  ✓ 20160921 - 48475 chars
  ✓ 20161214 - 39249 chars
  ✓ 20170315 - 40949 chars
  ✓ 20170614 - 45481 chars
  ✓ 20170920 - 50383 chars
  ✓ 20171213 - 52863 chars
  ✓ 20180321 - 42523 chars
  ✓ 20180613 - 51844 chars
  ✓ 20180926 - 57371 chars
  ✓ 20181219 - 43077 chars
  ✓ 20190130 - 46173 chars
  ✓ 20190320 - 43705 chars
  ✓ 20190501 - 39000 chars
  ✓ 20190619 - 42294 chars
  ✓ 2019073

In [ ]:
def split_transcript(text):
    if text is None:
        return None, None

    split_patterns = [
        r"(?i)i('d| would) be happy to take your questions",
        r"(?i)i('ll| will) now take your questions",
        r"(?i)i look forward to your questions",
        r"(?i)let me now take your questions",
        r"(?i)happy to take questions",
        r"(?i)take your questions",
    ]

    for pattern in split_patterns:
        match = re.search(pattern, text)
        if match:
            prepared = text[:match.end()].strip()
            qa = text[match.end():].strip()
            return prepared, qa

    reporter_match = re.search(r'\n[A-Z]{2,}\s[A-Z]{2,}\.?\s', text[500:])
    if reporter_match:
        split_point = 500 + reporter_match.start()
        return text[:split_point].strip(), text[split_point:].strip()

    return text, ""

split_data = {}
for date_str, text in transcripts.items():
    if text:
        prepared, qa = split_transcript(text)
        split_data[date_str] = {"prepared": prepared, "qa": qa}
        prep_pct = round(100 * len(prepared) / (len(prepared) + len(qa))) if qa else 100
        print(f"{date_str}: Prepared={prep_pct}% | Q&A={100-prep_pct}%")

print(f"\nSplit {len(split_data)} transcripts")

20110427: Prepared=1% | Q&A=99%
20110622: Prepared=1% | Q&A=99%
20111102: Prepared=20% | Q&A=80%
20120125: Prepared=1% | Q&A=99%
20120425: Prepared=10% | Q&A=90%
20120620: Prepared=12% | Q&A=88%
20121212: Prepared=18% | Q&A=82%
20130320: Prepared=16% | Q&A=84%
20130619: Prepared=21% | Q&A=79%
20130918: Prepared=25% | Q&A=75%
20131218: Prepared=18% | Q&A=82%
20140319: Prepared=24% | Q&A=76%
20140618: Prepared=23% | Q&A=77%
20140917: Prepared=33% | Q&A=67%
20141217: Prepared=23% | Q&A=77%
20150318: Prepared=23% | Q&A=77%
20150617: Prepared=22% | Q&A=78%
20150917: Prepared=26% | Q&A=74%
20151216: Prepared=27% | Q&A=73%
20160316: Prepared=26% | Q&A=74%
20160615: Prepared=24% | Q&A=76%
20160921: Prepared=19% | Q&A=81%
20161214: Prepared=18% | Q&A=82%
20170315: Prepared=18% | Q&A=82%
20170614: Prepared=25% | Q&A=75%
20170920: Prepared=19% | Q&A=81%
20171213: Prepared=16% | Q&A=84%
20180321: Prepared=15% | Q&A=85%
20180613: Prepared=19% | Q&A=81%
20180926: Prepared=12% | Q&A=88%
20181219: Pre

In [ ]:
analyzer = SentimentIntensityAnalyzer()

# Word lists
cautious_words = [
    "may", "might", "could", "possibly", "perhaps", "uncertain",
    "somewhat", "broadly", "roughly", "approximately", "monitor",
    "watching", "depends", "evaluate", "assess", "gradual",
    "cautious", "patient", "tentative", "moderate", "evolving",
    "unclear", "we'll see", "data dependent", "on the table",
    "range of outcomes", "both sides", "balanced", "measured",
    "some", "fairly", "reasonably", "appears", "seems",
    "suggest", "indicate", "not yet", "premature"
]

definitive_words = [
    "will", "committed", "confident", "clearly", "strongly",
    "determined", "certain", "absolutely", "must", "decisive",
    "without question", "no doubt", "firmly", "fully",
    "unequivocally", "precisely", "exactly", "certainly",
    "indeed", "necessary", "essential", "critical",
    "significant", "substantial", "robust", "solid",
    "convinced", "unwavering", "resolute"
]

jargon_words = [
    "quantitative", "tightening", "easing", "disinflationary",
    "transitory", "forward guidance", "dot plot", "neutral rate",
    "terminal rate", "balance sheet", "tapering", "basis points",
    "dovish", "hawkish", "yield curve", "liquidity", "repo",
    "overnight", "federal funds"
]

def count_words(text, word_list):
    text_lower = text.lower()
    count = 0
    for word in word_list:
        count += len(re.findall(r'\b' + word + r'\b', text_lower))
    return count

def avg_sentence_length(text):
    sentences = re.split(r'[.!?]+', text)
    sentences = [s.strip() for s in sentences if len(s.strip()) > 5]
    if not sentences:
        return 0
    return np.mean([len(s.split()) for s in sentences])

def sentence_sentiment(text):
    if not text:
        return 0
    sentences = re.split(r'[.!?]+', text)
    sentences = [s.strip() for s in sentences if len(s.strip()) > 10]
    if not sentences:
        return 0
    scores = [analyzer.polarity_scores(s)["compound"] for s in sentences]
    return round(np.mean(scores), 4)

def get_chair(date_str):
    dt = datetime.strptime(date_str, "%Y-%m-%d")
    if dt < datetime(2014, 2, 3):
        return "Bernanke"
    elif dt < datetime(2018, 2, 5):
        return "Yellen"
    else:
        return "Powell"

rows = []
for date_str, data in split_data.items():
    full = data["prepared"] + " " + data["qa"]
    prepared = data["prepared"]
    qa = data["qa"]

    total_words = len(full.split())
    cautious_count = count_words(full, cautious_words)
    definitive_count = count_words(full, definitive_words)
    jargon_count = count_words(full, jargon_words)

    cautious_ratio = round(cautious_count / max(definitive_count, 1), 3)
    jargon_freq = round(jargon_count / max(total_words, 1) * 1000, 3)

    fk_grade = flesch_kincaid_grade(full)
    sent_length = avg_sentence_length(full)

    prep_sent = sentence_sentiment(prepared)
    qa_sent = sentence_sentiment(qa)
    tone_shift = round(prep_sent - qa_sent, 4)

    date_fmt = datetime.strptime(date_str, "%Y%m%d").strftime("%Y-%m-%d")

    rows.append({
        "date": date_fmt,
        "chair": get_chair(date_fmt),
        "fk_grade": round(fk_grade, 2),
        "avg_sentence_length": round(sent_length, 2),
        "total_words": total_words,
        "cautious_count": cautious_count,
        "definitive_count": definitive_count,
        "cautious_ratio": cautious_ratio,
        "jargon_count": jargon_count,
        "jargon_freq": jargon_freq,
        "prepared_sentiment": prep_sent,
        "qa_sentiment": qa_sent,
        "tone_shift": tone_shift,
    })

linguistic_df = pd.DataFrame(rows).sort_values("date").reset_index(drop=True)

print(f"{len(linguistic_df)} rows")
print(f"\nChair breakdown:")
print(linguistic_df["chair"].value_counts())
print(f"\nSentiment range check:")
print(f"  Prepared: {linguistic_df['prepared_sentiment'].min():.4f} to {linguistic_df['prepared_sentiment'].max():.4f}")
print(f"  Q&A: {linguistic_df['qa_sentiment'].min():.4f} to {linguistic_df['qa_sentiment'].max():.4f}")
print(f"  Tone shift: {linguistic_df['tone_shift'].min():.4f} to {linguistic_df['tone_shift'].max():.4f}")

linguistic_df.head(10)

90 rows

Chair breakdown:
chair
Powell      63
Yellen      16
Bernanke    11
Name: count, dtype: int64

Sentiment range check:
  Prepared: 0.0214 to 0.2956
  Q&A: 0.0237 to 0.1824
  Tone shift: -0.0938 to 0.2066


,date,chair,fk_grade,avg_sentence_length,total_words,cautious_count,definitive_count,cautious_ratio,jargon_count,jargon_freq,prepared_sentiment,qa_sentiment,tone_shift
0,2011-04-27,Bernanke,11.60,18.57,8681,78,92,0.848,28,3.225,0.2521,0.1331,0.1190
1,2011-06-22,Bernanke,10.94,18.11,7306,87,49,1.776,7,0.958,0.2337,0.0765,0.1572
2,2011-11-02,Bernanke,11.48,17.87,7402,69,58,1.190,8,1.081,0.1036,0.1149,-0.0113
3,2012-01-25,Bernanke,13.78,23.56,10066,112,87,1.287,23,2.285,0.1947,0.1824,0.0123
4,2012-04-25,Bernanke,11.55,18.93,7644,96,57,1.684,28,3.663,0.0214,0.1152,-0.0938
5,2012-06-20,Bernanke,10.85,17.61,7239,87,44,1.977,10,1.381,0.1038,0.1139,-0.0101
6,2012-12-12,Bernanke,12.25,20.77,11611,125,119,1.050,39,3.359,0.1228,0.1234,-0.0006
7,2013-03-20,Bernanke,10.43,17.21,8906,113,63,1.794,28,3.144,0.1800,0.0583,0.1217
8,2013-06-19,Bernanke,11.07,19.01,9414,113,73,1.548,32,3.399,0.2329,0.1384,0.0945
9,2013-09-18,Bernanke,12.06,20.35,9014,100,80,1.250,43,4.770,0.1967,0.1605,0.0362


In [ ]:
sp500 = yf.download("^GSPC", start="2011-01-01", end="2026-04-01")
vix = yf.download("^VIX", start="2011-01-01", end="2026-04-01")

if isinstance(sp500.columns, pd.MultiIndex):
    sp500.columns = sp500.columns.get_level_values(0)
if isinstance(vix.columns, pd.MultiIndex):
    vix.columns = vix.columns.get_level_values(0)

sp500 = sp500.reset_index()
vix = vix.reset_index()

market_rows = []
for date_str in linguistic_df["date"]:
    dt = pd.Timestamp(date_str)

    sp_day = sp500[sp500["Date"] == dt]
    vix_day = vix[vix["Date"] == dt]

    future = sp500[sp500["Date"] > dt].head(1)

    if sp_day.empty or future.empty:
        print(f"  ✗ {date_str} - no market data")
        market_rows.append({"date": date_str})
        continue

    same_day_open = sp_day["Open"].values[0]
    same_day_close = sp_day["Close"].values[0]
    same_day_return = (same_day_close - same_day_open) / same_day_open * 100
    same_day_abs = abs(same_day_return)

    next_day_open = future["Open"].values[0]
    next_day_close = future["Close"].values[0]
    next_day_return = (next_day_close - next_day_open) / next_day_open * 100

    reversal = 1 if (same_day_return * next_day_return < 0) else 0

    vix_close = vix_day["Close"].values[0] if not vix_day.empty else np.nan

    market_rows.append({
        "date": date_str,
        "sp500_same_day_return": round(same_day_return, 4),
        "sp500_same_day_abs": round(same_day_abs, 4),
        "sp500_next_day_return": round(next_day_return, 4),
        "reversal": reversal,
        "vix_close": round(vix_close, 2) if not np.isnan(vix_close) else np.nan,
    })
    print(f"  ✓ {date_str} | Return: {same_day_return:.2f}% | Reversal: {'Yes' if reversal else 'No'}")

market_df = pd.DataFrame(market_rows)
print(f"\n{len(market_df)} rows")

/tmp/ipykernel_680/2033899836.py:1: FutureWarning: YF.download() has changed argument auto_adjust default to True
  sp500 = yf.download("^GSPC", start="2011-01-01", end="2026-04-01")
[*********************100%***********************]  1 of 1 completed
/tmp/ipykernel_680/2033899836.py:2: FutureWarning: YF.download() has changed argument auto_adjust default to True
  vix = yf.download("^VIX", start="2011-01-01", end="2026-04-01")
[*********************100%***********************]  1 of 1 completed


  ✓ 2011-04-27 | Return: 0.54% | Reversal: No
  ✓ 2011-06-22 | Return: -0.64% | Reversal: No
  ✓ 2011-11-02 | Return: 1.50% | Reversal: No
  ✓ 2012-01-25 | Return: 0.89% | Reversal: Yes
  ✓ 2012-04-25 | Return: 1.35% | Reversal: No
  ✓ 2012-06-20 | Return: -0.17% | Reversal: No
  ✓ 2012-12-12 | Return: 0.04% | Reversal: Yes
  ✓ 2013-03-20 | Return: 0.67% | Reversal: Yes
  ✓ 2013-06-19 | Return: -1.39% | Reversal: No
  ✓ 2013-09-18 | Return: 1.16% | Reversal: Yes
  ✓ 2013-12-18 | Return: 1.64% | Reversal: No
  ✓ 2014-03-19 | Return: -0.61% | Reversal: Yes
  ✓ 2014-06-18 | Return: 0.73% | Reversal: No
  ✓ 2014-09-17 | Return: 0.11% | Reversal: No
  ✓ 2014-12-17 | Return: 1.98% | Reversal: No
  ✓ 2015-03-18 | Return: 1.29% | Reversal: Yes
  ✓ 2015-06-17 | Return: 0.14% | Reversal: No
  ✓ 2015-09-17 | Return: -0.26% | Reversal: No
  ✓ 2015-12-16 | Return: 1.30% | Reversal: Yes
  ✓ 2016-03-16 | Return: 0.64% | Reversal: No
  ✓ 2016-06-15 | Return: -0.29% | Reversal: Yes
  ✓ 2016-09-21 | Ret

In [ ]:
# Merge linguistic and market data
df = linguistic_df.merge(market_df, on="date", how="left")

# Drop rows with missing market data
df = df.dropna(subset=["sp500_same_day_return"]).reset_index(drop=True)

# Preview
print(f"Final dataset: {len(df)} rows x {len(df.columns)} columns")
print(f"\nColumns:\n{list(df.columns)}")
print(f"\nReversal breakdown:")
print(df["reversal"].value_counts())
print(f"\nBasic stats:")
print(df.describe())

df.head(10)

Final dataset: 89 rows x 18 columns

Columns:
['date', 'chair', 'fk_grade', 'avg_sentence_length', 'total_words', 'cautious_count', 'definitive_count', 'cautious_ratio', 'jargon_count', 'jargon_freq', 'prepared_sentiment', 'qa_sentiment', 'tone_shift', 'sp500_same_day_return', 'sp500_same_day_abs', 'sp500_next_day_return', 'reversal', 'vix_close']

Reversal breakdown:
reversal
0.0    50
1.0    39
Name: count, dtype: int64

Basic stats:
        fk_grade  avg_sentence_length   total_words  cautious_count  \
count  89.000000            89.000000     89.000000       89.000000   
mean    9.868315            16.607978   8509.595506       88.292135   
std     1.384099             2.378464   1247.922398       18.819237   
min     7.960000            13.160000   2347.000000       17.000000   
25%     8.750000            14.720000   7644.000000       76.000000   
50%     9.430000            15.990000   8615.000000       87.000000   
75%    11.030000            18.110000   9366.000000      100.00

,date,chair,fk_grade,avg_sentence_length,total_words,cautious_count,definitive_count,cautious_ratio,jargon_count,jargon_freq,prepared_sentiment,qa_sentiment,tone_shift,sp500_same_day_return,sp500_same_day_abs,sp500_next_day_return,reversal,vix_close
0,2011-04-27,Bernanke,11.60,18.57,8681,78,92,0.848,28,3.225,0.2521,0.1331,0.1190,0.5362,0.5362,0.4890,0.0,15.35
1,2011-06-22,Bernanke,10.94,18.11,7306,87,49,1.776,7,0.958,0.2337,0.0765,0.1572,-0.6438,0.6438,-0.2409,0.0,18.52
2,2011-11-02,Bernanke,11.48,17.87,7402,69,58,1.190,8,1.081,0.1036,0.1149,-0.0113,1.4988,1.4988,1.8494,0.0,32.74
3,2012-01-25,Bernanke,13.78,23.56,10066,112,87,1.287,23,2.285,0.1947,0.1824,0.0123,0.8871,0.8871,-0.5919,1.0,18.31
4,2012-04-25,Bernanke,11.55,18.93,7644,96,57,1.684,28,3.663,0.0214,0.1152,-0.0938,1.3541,1.3541,0.6716,0.0,16.82
5,2012-06-20,Bernanke,10.85,17.61,7239,87,44,1.977,10,1.381,0.1038,0.1139,-0.0101,-0.1731,0.1731,-2.2074,0.0,17.24
6,2012-12-12,Bernanke,12.25,20.77,11611,125,119,1.050,39,3.359,0.1228,0.1234,-0.0006,0.0448,0.0448,-0.6321,1.0,15.95
7,2013-03-20,Bernanke,10.43,17.21,8906,113,63,1.794,28,3.144,0.1800,0.0583,0.1217,0.6697,0.6697,-0.8282,1.0,12.67
8,2013-06-19,Bernanke,11.07,19.01,9414,113,73,1.548,32,3.399,0.2329,0.1384,0.0945,-1.3863,1.3863,-2.2424,0.0,16.64
9,2013-09-18,Bernanke,12.06,20.35,9014,100,80,1.250,43,4.770,0.1967,0.1605,0.0362,1.1596,1.1596,-0.2895,1.0,13.59


In [ ]:
df.to_csv("fed_presser_dataset.csv", index=False)
from google.colab import files
files.download("fed_presser_dataset.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>